# 🔧 Auto-Import & GraphQL Internals

Advanced topics:
- **ImportClient** — GitHub, GitLab, and OSH auto-import
- **GraphQLClient** — direct queries, signing, instance variables
- **Config deep dive** — proxyUrl derivation, KeyStorage customization

## Setup

In [ ]:
(async () => {
  const { InterfacerClient, createConfig, deriveEndpointsFromProxy, createMemoryStorage } = await import('https://esm.sh/@dyne/interfacer-client');
  const { SANDBOX_CONFIG } = await import('/files/config.js');
  const client = new InterfacerClient(SANDBOX_CONFIG);
  console.log('🔧 Import & internals ready');
  console.log('🔐 Authenticated:', client.isAuthenticated());
})();


## 1. Import from GitHub

Auto-fetches repo metadata, license, and README from the GitHub API.

In [ ]:
(async () => {
  try {
    const data = await client.import.importFromGithub(
      'https://github.com/interfacerproject/interfacer-client'
    );
    
    console.log('✅ GitHub import:');
    console.log('  Title:', data.main?.title);
    console.log('  Description:', data.main?.description?.substring(0, 100) + '...');
    console.log('  Tags:', data.main?.tags?.join(', '));
    console.log('  License:', data.licenses?.map(l => l.licenseId).join(', '));
  } catch (err) {
    console.warn('⚠️ GitHub import failed (rate limit?):', err.message);
  }
})();


## 2. Import from GitLab

In [ ]:
(async () => {
  try {
    // Example: GitLab project ID for a public project
    const data = await client.import.importFromGitlab(
      'https://gitlab.com',
      'gitlab-org/gitlab'  // This may fail on private repos
    );
    console.log('✅ GitLab import:', data.main?.title);
  } catch (err) {
    console.warn('⚠️ GitLab import failed (expected for private/rate-limited):', err.message);
  }
})();


## 3. Open Source Hardware Compliance Check

Analyzes a repository against OSH criteria.

In [ ]:
(async () => {
  try {
    const isOsh = await client.import.analyzeRepoForOsh(
      'https://github.com/interfacerproject/interfacer-client'
    );
    console.log('✅ OSH compliant:', isOsh);
  } catch (err) {
    console.warn('⚠️ OSH check failed (oshUrl may not be configured):', err.message);
  }
})();


### ImportedProjectData Structure

```typescript
interface ImportedProjectData {
  main?: {
    title: string;
    link: string;
    description: string;
    tags: string[];
  };
  licenses?: Array<{
    scope: string;      // "All", "Hardware", "Software"
    licenseId: string;  // SPDX identifier
  }>;
}
```

## 4. GraphQL Direct Access

The `GraphQLClient` handles signing and fetches. You can use it directly
for custom queries not covered by the sub-client methods.

In [ ]:
(async () => {
  // Direct GraphQL query (unauthenticated)
  const query = `
    query FetchInstanceVariables {
      instanceVariables {
        id
        projectDesign { id }
        projectProduct { id }
        projectService { id }
        machine { id }
        dpp { id }
        unitOne { id }
      }
    }
  `;
  try {
    const result = await client.graphql.request(query);
    console.log('✅ Direct GraphQL query succeeded');
    
    if (result.data) {
      console.log('  Data keys:', Object.keys(result.data).join(', '));
    }
    if (result.errors) {
      console.log('  Errors:', result.errors);
    }
  } catch (err) {
    console.warn('⚠️ GraphQL request failed:', err.message);
  }
  // Enable/disable signing
  console.log('\n📝 Signing status:', 'enabled after login');
  // Import signing functions directly
  const { signGraphQLRequest, signDidRequest, signFileUpload } = await import('https://esm.sh/@dyne/interfacer-client');
  console.log('🔏 Available signing functions:');
  console.log('  signGraphQLRequest(body, store) — for Zenflows API');
  console.log('  signDidRequest(body, store) — for DPP API');
  console.log('  signFileUpload(body, store) — for file operations');
})();


## 5. Instance Variables

Zenflows deployments expose global configuration through `instanceVariables`.
These are cached after first fetch — use `clearInstanceVariablesCache()` to refresh.

In [ ]:
(async () => {
  const { getInstanceVariables, clearInstanceVariablesCache } = await import('https://esm.sh/@dyne/interfacer-client');
  try {
    const vars = await getInstanceVariables(client.graphql);
    console.log('🔧 Instance variables:');
    console.log('  Design spec:', vars.projectDesign.id);
    console.log('  Product spec:', vars.projectProduct?.id);
    console.log('  Service spec:', vars.projectService?.id);
    console.log('  Machine spec:', vars.machine?.id);
    console.log('  DPP spec:', vars.dpp?.id);
    console.log('  Unit One:', vars.unitOne.id);
    
    // Clear cache (force re-fetch)
    // clearInstanceVariablesCache();
    // console.log('🔄 Cache cleared — next call will re-fetch');
  } catch (err) {
    console.warn('⚠️ Instance variables fetch failed:', err.message);
  }
})();


## 6. Config Deep Dive

Advanced configuration patterns.

In [ ]:
// deriveEndpointsFromProxy — see all derived URLs
const endpoints = deriveEndpointsFromProxy('https://proxy.dpp-dev.ddns.dyne.org');
console.log('🌐 Endpoints derived from proxyUrl:');
console.log(JSON.stringify(endpoints, null, 2));
// createConfig — merge explicit values with derived endpoints
const customConfig = createConfig({
  proxyUrl: 'https://proxy.dpp-dev.ddns.dyne.org',
  // Override specific endpoint
  zenflowsUrl: 'https://custom-zenflows.example.com/api',
  zenflowsAdmin: 'secret-token',  // Not derived from proxy
});
console.log('\n⚙️ Custom config:');
console.log('  proxyUrl:', customConfig.proxyUrl);
console.log('  zenflowsUrl:', customConfig.zenflowsUrl, '(overridden)');
console.log('  dppUrl:', customConfig.dppUrl, '(derived)');
console.log('  zenflowsAdmin:', customConfig.zenflowsAdmin, '(explicit)');
// KeyStorage: In-memory (non-persistent)
const memStore = createMemoryStorage();
const ephemeral = new InterfacerClient(SANDBOX_CONFIG, memStore);
console.log('\n💾 Memory storage client ready');
console.log('  isAuthenticated:', ephemeral.isAuthenticated());
// Config fields that CANNOT be derived from proxyUrl:
console.log('\n📌 Fields requiring explicit config:');
console.log('  loshId, zenflowsAdmin, specs (dpp/machine/material/product/service)');
console.log('  walletCycle ({ startDate, cycleLength }), location ({ autocomplete, lookup })');


## 7. SDK Export Map

Complete list of all publicly exported symbols from `@dyne/interfacer-client`:

In [ ]:
(async () => {
  const sdk = await import('https://esm.sh/@dyne/interfacer-client');
  console.log('📦 All exports (' + Object.keys(sdk).length + ' total):');
  console.log('\n🔷 Main facade:');
  console.log('  InterfacerClient');
  console.log('\n🔷 Sub-clients:');
  console.log('  AuthClient, ResourceClient, FileClient, DppClient');
  console.log('  InboxClient, WalletClient, SocialClient, TaggingClient, ImportClient');
  console.log('\n🔷 GraphQL:');
  console.log('  GraphQLClient, getInstanceVariables, clearInstanceVariablesCache');
  console.log('\n🔷 Config:');
  console.log('  createConfig, deriveEndpointsFromProxy, createMemoryStorage, createLocalStorageAdapter');
  console.log('\n🔷 Crypto:');
  console.log('  signGraphQLRequest, signDidRequest, signFileUpload');
  console.log('\n🔷 Files:');
  console.log('  prepFileForZenflows, prepFilesForZenflows, formatImageSrc, getResourceImage');
  console.log('\n🔷 Tagging:');
  console.log('  slugifyTagValue, prefixedTag, userTag, isUserTag, stripUserTagPrefix,');
  console.log('  isSystemTag, extractUserTagValues, normalizeUserTagsForSave,');
  console.log('  monotonicRangeTags, rangeFilterTags, derivedProductFilterTags,');
  console.log('  mergeTags, removeTagsWithPrefixes');
  console.log('\n🔷 Constants:');
  console.log('  TAG_PREFIX, SYSTEM_TAG_PREFIXES, MANUFACTURABLE_TRUE_TAG,');
  console.log('  REPAIRABILITY_AVAILABLE_TAG, PRODUCT_CATEGORY_OPTIONS,');
  console.log('  POWER_COMPATIBILITY_OPTIONS, REPLICABILITY_OPTIONS,');
  console.log('  SERVICE_TYPE_OPTIONS, AVAILABILITY_OPTIONS,');
  console.log('  RECYCLABILITY_THRESHOLDS_PCT, POWER_REQUIREMENT_THRESHOLDS_W,');
  console.log('  ENERGY_THRESHOLDS_KWH, CO2_THRESHOLDS_KG');
  console.log('\n🔷 Domain types:');
  console.log('  Project, ProjectType, ProductFilters, ServiceFilters, License,');
  console.log('  Contributor, Proposal, MachineRef, MaterialRef, ImageRef, ModelFile, Location');
  console.log('  UserChallenges, UserProfile, Keyring, KeyStorage, InterfacerConfig');
})();


## 📋 API Reference

### ImportClient

| Method | Description |
|---|---|
| `importFromGithub(url)` | Fetch repo metadata + license from GitHub API |
| `importFromGitlab(host, projectId)` | Fetch project metadata from GitLab API |
| `analyzeRepoForOsh(repoUrl)` | Check OSH compliance via oshUrl endpoint |

### GraphQLClient

| Method | Description |
|---|---|
| `request(query, variables?, extraHeaders?)` | Execute any GraphQL operation |
| `setSigningEnabled(bool)` | Enable EdDSA request signing |

## ✅ Summary
- ✅ GitHub & GitLab auto-import with metadata extraction
- ✅ OSH compliance analysis
- ✅ Direct GraphQL access with signing control
- ✅ Instance variables (cached, clearable)
- ✅ Config deep dive (proxyUrl, explicit override, KeyStorage)
- ✅ Complete SDK export map

## 🎉 You've completed the full Interfacer SDK documentation!

| Notebook | Topic |
|---|---|
| `00_Quick_Start` | Setup, architecture, config |
| `01_Authentication_and_Crypto` | Keypairoom auth, EdDSA |
| `02_Resource_Management` | Projects, machines, proposals |
| `03_Files_and_Hashing` | SHA-512, SHA-256, uploads |
| `04_DPP` | Digital Product Passport |
| `05_Social_and_Inbox` | Messaging, ActivityPub |
| `06_Wallet_and_Tagging` | Points, classification |
| `07_Import_and_Internals` | Auto-import, GraphQL, config |